In [1]:
import numpy as np
import pandas as pd
import umap
import hdbscan
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity

# Set a clean, professional aesthetic for all plots
sns.set_theme(style="white")
# Define consistent colors for the entire presentation
COLORS = {'Human': '#1f77b4', 'Agent': '#d62728'}

from google.colab import drive

# 1. Mount Drive (Essential for accessing your saved data.zip)
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
DESTINATION_DIR = '/content/drive/MyDrive/Colab Notebooks/data/cds'

In [3]:
df = pd.read_pickle(f"{DESTINATION_DIR}/final_merged_15-3.pkl")

In [4]:
df["interaction_type"].value_counts()

,count
interaction_type,
comment,36743
submission,7626
post,7626


In [ ]:
# Standardize interaction types
df["interaction_type"] = df["interaction_type"].replace("submission", "post")

# Stack embeddings safely into a 2D array
embeddings = np.stack(df["embedding"].values)

# Map labels (Applying Fix 1: 0 -> Agent, 1 -> Human based on your notebook)
labels = df['label'].map({0: 'Agent', 1: 'Human'}).values

# Parameters for sweep
neighbors_to_test = [15, 50, 100, 200]
min_dist_val = 0.05

# Set up the Matplotlib 2x2 grid
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle("UMAP Parameter Sweep: Finding the Best Global Structure", fontsize=16, fontweight='bold')
axes = axes.flatten()

for i, n in enumerate(neighbors_to_test):
    print(f"Fitting UMAP with n_neighbors={n}...")
    
    reducer = umap.UMAP(n_neighbors=n, min_dist=min_dist_val, n_components=2, random_state=42)
    embedding_2d = reducer.fit_transform(embeddings)
    
    df_temp = pd.DataFrame(embedding_2d, columns=['x', 'y'])
    df_temp['Label'] = labels
    
    # Plot Human
    human_data = df_temp[df_temp['Label'] == 'Human']
    axes[i].scatter(human_data['x'], human_data['y'], c=COLORS['Human'], label='Human', alpha=0.4, s=3, edgecolors='none')
    
    # Plot Agent
    agent_data = df_temp[df_temp['Label'] == 'Agent']
    axes[i].scatter(agent_data['x'], agent_data['y'], c=COLORS['Agent'], label='Agent', alpha=0.4, s=3, edgecolors='none')
    
    axes[i].set_title(f"n_neighbors = {n}")
    axes[i].set_xticks([])
    axes[i].set_yticks([])
    if i == 0:
        axes[i].legend(markerscale=3)

plt.tight_layout()
plt.savefig("umap_parameter_sweep_matplotlib.png", dpi=300)
plt.show()

Fitting UMAP with n_neighbors=15...


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.12/dist-packages/umap/spectral.py:548: UserWarning: Spectral initialisation failed! The eigenvector solver
failed. This is likely due to too small an eigengap. Consider
adding some noise or jitter to your data.

Falling back to random initialisation!
  warn(
/usr/local/lib/python3.12/dist-packages/umap/spectral.py:548: UserWarning: Spectral initialisation failed! The eigenvector solver
failed. This is likely due to too small an eigengap. Consider
adding some noise or jitter to your data.

Falling back to random initialisation!
  warn(
/usr/local/lib/python3.12/dist-packages/umap/spectral.py:548: UserWarning: Spectral initialisation failed! The eigenvector solver
failed. This is likely due to too small an eigengap. Consider
adding some noise or jitter to your data.

Falling back to random initialisati

Fitting UMAP with n_neighbors=50...


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [ ]:
print("Fitting final UMAP on all rows... Establishing global coordinate space.")
reducer = umap.UMAP(n_neighbors=50, min_dist=0.05, n_components=2, random_state=42)
embedding_2d = reducer.fit_transform(embeddings)

# Prepare Master DataFrame with Corrected Mapping (Fix 1)
df_viz = pd.DataFrame(embedding_2d, columns=['UMAP_1', 'UMAP_2'])
df_viz['Label'] = df['label'].map({0: 'Agent', 1: 'Human'}).values
df_viz['interaction_type'] = df['interaction_type'].values

SyntaxError: unmatched ')' (2520087830.py, line 1)

In [ ]:
print("Running HDBSCAN...")
clusterer = hdbscan.HDBSCAN(min_cluster_size=50, gen_min_span_tree=True)
cluster_labels = clusterer.fit_predict(embeddings)

# Prepare DataFrame for Visualization
df_viz_cluster = pd.DataFrame(embedding_2d, columns=['UMAP_1', 'UMAP_2'])
df_viz_cluster['Cluster'] = cluster_labels.astype(str)
df_viz_cluster['Label'] = df['label'].map({0: 'Agent', 1: 'Human'}).values

# Fallback Fix (Fix 2: Safely handle source column)
if 'source' in df.columns:
    df_viz_cluster['Source'] = df['source'].values
else:
    df_viz_cluster['Source'] = 'Unknown'

# Clean up noise labels for a better legend
df_viz_cluster['Cluster'] = df_viz_cluster['Cluster'].replace('-1', 'Noise (Unclustered)')

plt.figure(figsize=(12, 10))

# Seaborn scatterplot for beautiful multi-category styling
sns.scatterplot(
    data=df_viz_cluster, 
    x='UMAP_1', 
    y='UMAP_2', 
    hue='Cluster', 
    style='Label', # Differentiates Agent vs Human by marker shape
    palette='tab20', # Uses a diverse 20-color palette for many clusters
    alpha=0.6,
    s=20,
    edgecolor=None
)

plt.title("Graph C: UMAP Projection with HDBSCAN Clustering", fontsize=14, fontweight='bold')
# Move the legend outside the plot so it doesn't cover your data
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0)
plt.tight_layout()
plt.savefig("graph_c_hdbscan_matplotlib.png", dpi=300, bbox_inches='tight')
plt.show()